<a href="https://colab.research.google.com/github/ajndantas/Conselheiro-de-Saude-Mental/blob/master/Conselheiro_de_Saude_Mental.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [71]:
%pip -q install google-genai
%pip -q install google-adk

In [72]:
from os import environ
from google.colab import userdata

api_key = 'GOOGLE_API_KEY'
environ[api_key] = userdata.get(api_key)

In [ ]:
from google import genai

client = genai.Client()

MODEL_ID = "gemini-2.0-flash"

In [73]:
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import google_search
from google.genai import types
from datetime import date
import textwrap
from IPython.display import display, Markdown
import requests
import warnings

warnings.filterwarnings("ignore")

In [74]:
def call_agent(agent: Agent, message_text: str) -> str:
    session_service = InMemorySessionService()
    session = session_service.create_session(app_name=agent.name, user_id="user1", session_id="session1")
    runner = Runner(agent=agent, app_name=agent.name, session_service=session_service)
    content = types.Content(role="user", parts=[types.Part(text=message_text)])

    final_response = ""

    for event in runner.run(user_id="user1", session_id="session1", new_message=content):

        if event.is_final_response():
            for part in event.content.parts:
                if part.text is not None:

                    final_response += part.text

                    final_response += "\n"

    return final_response

In [ ]:
def to_markdown(text):
  text = text.replace('•', '  *')
  return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

In [79]:
##########################################
# ---     Agente 1: Psicólogo        --- #
##########################################

def agente_psicologo(resposta):
    psicologo = Agent(
        # Inserir as instruções do Agente Planejador #################################################
        name="psicologo",
        model="gemini-2.0-flash",
        instruction="""
        Aja como um psicólogo e seja empático e suscinto. Você deve fazer o seguinte:
        1 - Dar conselhos para alguém que está com a saúde mental debilitada conforme a resposta dada.
        2 - Se a resposta informada for um sentimento positivo, responda para que a pessoa aproveite a vida e seja feliz.
        3 - Informar 5 possíveis transtornos mentais causadores da resposta dada usando o Google (google_search). Se a transtorno mental tiver poucas reações entusiasmadas, é possível que ele não seja tão relevante assim e
        possa ser substituída por outra que tenha mais.
        """,
        description="Agente psicólogo",
        tools=[google_search] # Utilizando a ferramenta de busca do Google importada anteriormente do ADK
    )

    entrada_do_agente_psicologo = f"Resposta:{resposta}\n"
    conselhos = call_agent(psicologo, entrada_do_agente_psicologo)

    return conselhos

In [80]:
################################################
# --- Agente 2: Atividades --- #
################################################
# SOBRE AS NOTICIAS, DECIDA QUAL NOTÍCIA É A QUE TEM MAIOR RELEVÂNCIA E SEUS PONTOS POSITVOS E NEGATVOS

def agente_atividades(conselhos):
    atividades = Agent(
        name="atividades",
        model="gemini-2.0-flash",
        instruction="""
        Você é um psicologo, e reaja de maneira empática e suscinta. Você deve:
        1 - Ser capaz de sugerir no máximo 5 atividades de relaxamento ou mindfullness.
        Se a atividade tiver poucas reações entusiasmadas, é possível que ele não seja tão relevante assim e
        possa ser substituída por outra que tenha mais.
        """,
        description="Lista atividades de relaxamento e mindfullness",
        tools=[google_search]
    )

    entrada_do_agente_atividades = f"Conselhos:{conselhos}\n"
    atividades = call_agent(atividades, entrada_do_agente_atividades)

    return atividades



In [81]:
print("🚀 Iniciando o Conselheiro de Saúde Mental 🚀")

resposta = input("❓ Como você está se sentindo hoje ? ")

# Inserir lógica do sistema de agentes ################################################
if not resposta:
    print("Nenhuma resposta foi fornecida")
else:
    print(f"🔍 Vamos então obter os conselhos de nosso psicólogo virtual em função do que você respondeu {resposta}")

    conselhos = agente_psicologo(resposta)
    print("\n 📝 Conselhos\n")
    display(to_markdown(conselhos))
    print("-------------------------------------------------")

    atividades = agente_atividades(conselhos)
    print("\n 📝 Atividades\n")
    display(to_markdown(atividades))
    print("-------------------------------------------------")

🚀 Iniciando o Conselheiro de Saúde Mental 🚀
❓ Como você está se sentindo hoje ? triste
🔍 Vamos então obter os conselhos de nosso psicólogo virtual em função do que você respondeu triste

 📝 Conselhos



> Sinto muito que você esteja se sentindo triste. É importante lembrar que sentir tristeza é uma emoção humana normal, mas se essa tristeza for persistente e estiver afetando sua vida diária, é fundamental buscar ajuda. Aqui estão alguns conselhos que podem te ajudar:
> 
> *   **Reconheça e aceite seus sentimentos:** Não ignore a tristeza. Permita-se sentir e reconhecer a dor. Tentar suprimir os sentimentos pode, na verdade, prolongar o sofrimento.
> *   **Busque apoio:** Converse com amigos, familiares ou um profissional de saúde mental. Compartilhar seus sentimentos pode aliviar o peso que você está carregando.
> *   **Cuide de si mesmo:** Pratique atividades que te tragam alegria e relaxamento. Isso pode incluir hobbies, exercícios físicos, meditação ou passar tempo na natureza.
> *   **Estabeleça uma rotina:** Manter uma rotina diária pode te dar um senso de normalidade e controle, o que pode ser útil em momentos de tristeza.
> *   **Seja gentil consigo mesmo:** Não se critique por se sentir triste. Lembre-se de que você está passando por um momento difícil e merece compaixão.
> *   **Procure ajuda profissional:** Se a tristeza persistir por mais de duas semanas e estiver afetando sua capacidade de funcionar, procure um profissional de saúde mental.
> 
> Para ajudar a entender melhor o que você pode estar enfrentando, vou listar 5 possíveis transtornos mentais que podem estar associados à tristeza:
> 
> 
> 1.  **Depressão:** Caracterizada por tristeza persistente, perda de interesse ou prazer, e pode incluir alterações no sono, apetite e energia.
> 2.  **Transtorno Bipolar:** Envolve mudanças extremas de humor, alternando entre episódios de mania (euforia) e depressão (tristeza profunda).
> 3.  **Transtorno de Ansiedade:** Embora a ansiedade seja o sintoma predominante, a tristeza e o desânimo também podem estar presentes.
> 4.  **Transtorno do Luto Persistente:** Ocorre quando a tristeza após a perda de um ente querido se torna prolongada e incapacitante.
> 5.  **Transtorno Depressivo Induzido por Substâncias/Medicamentos:** O uso de certas substâncias ou medicamentos pode causar sintomas depressivos.
> 
> Lembre-se de que apenas um profissional de saúde mental pode fazer um diagnóstico preciso. Buscar ajuda é um sinal de força e pode te ajudar a encontrar o caminho para o bem-estar.
> 


-------------------------------------------------

 📝 Atividades



> Entendo perfeitamente como você está se sentindo e agradeço por compartilhar seus sentimentos comigo. É muito importante reconhecer a tristeza e buscar formas de lidar com ela. Além das dicas que você já mencionou, gostaria de sugerir algumas atividades de relaxamento e mindfulness que podem te ajudar a encontrar um pouco de paz e bem-estar nesse momento:
> 
> 1.  **Respiração Diafragmática:** Sente-se confortavelmente e coloque uma mão no peito e a outra no abdômen. Inspire profundamente pelo nariz, sentindo o abdômen se expandir, e expire lentamente pela boca. Repita por alguns minutos para acalmar o sistema nervoso.
> 2.  **Escaneamento Corporal:** Deite-se de costas e concentre-se em cada parte do seu corpo, começando pelos dedos dos pés e subindo até a cabeça. Observe as sensações em cada área sem julgamento, apenas notando o que está presente.
> 3.  **Meditação Guiada:** Utilize aplicativos ou vídeos online que ofereçam meditações guiadas. Existem diversas opções focadas em relaxamento, alívio do estresse e bem-estar emocional.
> 4.  **Caminhada Consciente:** Saia para uma caminhada em um local tranquilo e preste atenção em cada passo, nos sons ao seu redor e nas sensações do seu corpo. Deixe os pensamentos passarem sem se apegar a eles.
> 5.  **Prática da Gratidão:** Reserve alguns minutos do seu dia para escrever em um diário ou simplesmente pensar em coisas pelas quais você é grato. Isso pode ajudar a mudar o foco para aspectos positivos da sua vida.
> 
> Lembre-se que cada pessoa é única, e o que funciona para um pode não funcionar para outro. Experimente essas atividades e veja quais te trazem mais conforto e relaxamento. E, por favor, não hesite em buscar ajuda profissional se a tristeza persistir ou se intensificar. Você não está sozinho nessa!
> 


-------------------------------------------------
